# Valid Outcomes of Positive Support Size Four

In [8]:
import numpy as np
import math

import chipsplitting as cs
import chipsplitting.hyperfield.utils as utils

from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter

## Proposition 5.29

In [3]:
def absolute(d, c):
    if c == 'd-1':
        return d-1
    if c == 'd-2':
        return d-2
    if c == 'd-3':
        return d-3
    if c == 'd-4':
        return d-4
    if c == 'd-0':
        return d
    return c

def rel(d, contraction_size, index):
    assert index < contraction_size or index > d - contraction_size
    return f"d-{d - index}" if index > contraction_size else index
    
def sign(x):
    return np.sign(x)
    
def is_contractable(p, contraction_size = 5):
    assert p.degree >= contraction_size * 3 - 1

    # check b
    for row in range(contraction_size):
        for col in range(contraction_size, p.degree - contraction_size - row + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, row)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, row)]):
                return False

    # check c
    for col in range(contraction_size):
        for row in range(contraction_size, p.degree - contraction_size - col + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(col, contraction_size)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(col, contraction_size)]):
                return False

    # check d1
    for j in range(contraction_size):
        for i in range(contraction_size, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False
                
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False

    # check d2
    for j in range(contraction_size):
        for i in range(contraction_size + 1, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False

    return True

In [4]:
%%time
base_types = ["diag", "row", "col"]
support_size = 4
for d in range(6,12):
    A = [PascalForm(d, mode, unit).to_hyperfield() for mode in base_types for unit in range(d + 1)]
    linear_system = HLinSystem(A)
    solutions = []
    solutions = linear_system.quick_solve_loop(support_size)
    print(f"Number of configurations for degree={d}: {len(solutions)}")
    if solutions:
        print(f"Solutions are {[[cs.utils.to_coordinate(index) for index in sol] for sol in solutions]}")
        print()
print()

Number of configurations for degree=6: 5
Solutions are [[(np.int64(3), np.int64(0)), (np.int64(1), np.int64(4)), (np.int64(0), np.int64(6)), (np.int64(5), np.int64(1))], [(np.int64(0), np.int64(3)), (np.int64(4), np.int64(1)), (np.int64(1), np.int64(5)), (np.int64(6), np.int64(0))], [(np.int64(1), np.int64(1)), (np.int64(0), np.int64(6)), (np.int64(3), np.int64(3)), (np.int64(6), np.int64(0))], [(np.int64(1), np.int64(1)), (np.int64(0), np.int64(5)), (np.int64(3), np.int64(3)), (np.int64(6), np.int64(0))], [(np.int64(1), np.int64(1)), (np.int64(5), np.int64(0)), (np.int64(0), np.int64(6)), (np.int64(3), np.int64(3))]]

Number of configurations for degree=7: 3
Solutions are [[(np.int64(3), np.int64(1)), (np.int64(1), np.int64(5)), (np.int64(0), np.int64(7)), (np.int64(7), np.int64(0))], [(np.int64(1), np.int64(3)), (np.int64(5), np.int64(1)), (np.int64(0), np.int64(7)), (np.int64(7), np.int64(0))], [(np.int64(1), np.int64(1)), (np.int64(3), np.int64(3)), (np.int64(0), np.int64(7)), (np.

## Proposition 5.48


In [5]:
def countValidConfigsForContractions(degree="even"):
    assert degree == "even" or degree == "odd", "degree must be 'even' or 'odd'"

    pos_support_size = 4
    contraction_size = 4
    d = contraction_size * 3 - 1 + (1 if degree == "even" else 2)
    base_types = ["diag", "row", "col"]
    A = [PascalForm(d, b, k) for b in base_types for k in range(contraction_size)] + [PascalForm(d, b, k) for b in base_types for k in range(d - contraction_size + 1, d + 1)]
    A = [p.to_hyperfield().contract(contraction_size) for p in A]  
    linear_system = HLinSystem(A)
    solutions = linear_system.quick_solve_loop(pos_support_size)
    
    return solutions


In [6]:
%%time
res1 = countValidConfigsForContractions("even")
print(f"Number of configurations for even d: {len(res1)}")

res2 = countValidConfigsForContractions("odd")
print(f"Number of configurations for odd d: {len(res2)}")

Number of configurations for even d: 0
Number of configurations for odd d: 0
CPU times: user 16.9 ms, sys: 1.42 ms, total: 18.3 ms
Wall time: 17.1 ms


## Theorem 5.49

In [55]:
degree = 6
configurations = [
    [-1,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  1,  1,  0,  0,  1,  0,  0,  0],
    [-1,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  1,
         0,  0,  0,  0,  1,  0,  0,  0,  0,  1,  0],
    [-1,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  1,  0,  0,  1,  0,  0,  0,  0,  1],
    [-1,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,
         0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  1],
    [-1,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  1,  0,  0,  1,  0,  0,  1]
]

failed = []

for row in configurations:
    support = [utils.to_coordinate(i) for i, x in enumerate(row) if x == 1 or x == -1]
    basis = list([0,3,4,5,6])
    P = pairing_matrix(degree, basis, support)
    if np.linalg.matrix_rank(P) < 5:
        failed.append(row)
        print(f"Pairing matrix (support={support}, basis={basis}) has not full rank")

Pairing matrix (support=[(0, 0), (3, 0), (1, 4), (0, 6), (5, 1)], basis=[0, 3, 4, 5, 6]) has not full rank
Pairing matrix (support=[(0, 0), (0, 3), (4, 1), (1, 5), (6, 0)], basis=[0, 3, 4, 5, 6]) has not full rank


In [58]:
failed = []
for row in failed:
    support = [utils.to_coordinate(i) for i, x in enumerate(row) if x == 1 or x == -1]
    basis = list([0,1,4,5,6])
    P = pairing_matrix(degree, basis, support)
    if np.linalg.matrix_rank(P) < 5:
        failed.append(row)
        print(f"Pairing matrix (support={support}, basis={basis}) has not full rank")

if not failed:
    print("Success")

Success


In [67]:
degree = 7
configurations = [
    [-1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,
         0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,
         0,  1],
    [-1,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  1,  0,  0,  0,  0,  0,
         0,  1],
    [-1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  1,  0,  0,  0,  0,  0,
         0,  1]
]

failed = []


for row in configurations:
    support = [utils.to_coordinate(i) for i, x in enumerate(row) if x == 1 or x == -1]
    basis = list([0,2,4,6,7])
    P = pairing_matrix(degree, basis, support)
    if np.linalg.matrix_rank(P) < 5:
        failed.append(row)
        print(f"Pairing matrix (support={support}, basis={basis}) has not full rank")

if not failed:
    print("Success")

Success
